# 03 – Knowledge Agent (RAG / Vector Search)

The **KnowledgeAgent** does semantic similarity search over governance documents (policies, runbooks, standards).  
In mock mode it uses a keyword-scored in-memory store — no pgvector/OpenAI needed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

In [ ]:
from agents.knowledge_agent import KnowledgeAgent
from core.base_agent import AgentRequest

agent = KnowledgeAgent()

## 1. Basic knowledge lookup

In [ ]:
req = AgentRequest(query='What is the GRR threshold policy?')
result = agent.execute(req)

print('Success   :', result.success)
print('Message   :', result.message)
print('Confidence:', result.confidence)
print('Sources   :', result.sources)

print('\nKnowledge results:')
for k in result.data.get('knowledge', []):
    print(f"  [{k['topic']}] {k['definition'][:100]}...")
    print(f"  source: {k['source']}\n")

## 2. Different governance queries

In [ ]:
queries = [
    'What is the data governance framework?',
    'How should incidents be reported?',
    'What are the bookings reconciliation standards?',
    'Explain the LTV calculation methodology',
    'What is the CAC payback policy?',
]

for q in queries:
    r = agent.execute(AgentRequest(query=q))
    docs = r.data.get('knowledge', [])
    print(f"Query: '{q}'")
    print(f"  => {len(docs)} docs, confidence={r.confidence}")
    if docs:
        print(f"  => top: [{docs[0]['topic']}]")
    print()

## 3. Relevance threshold filtering

In [ ]:
from agents.knowledge_agent import RELEVANCE_THRESHOLD
from services.pgvector.mock import NullVectorService

print(f'Relevance threshold: {RELEVANCE_THRESHOLD}')

# Check raw scores from the vector service
svc = NullVectorService()
results = svc.similarity_search('retention policy GRR churn', k=6)

print('\nRaw similarity scores:')
for doc, score in results:
    above = 'PASS' if score >= RELEVANCE_THRESHOLD else 'FILTERED'
    print(f'  {above} ({score})  {doc.metadata["topic"]}')

## 4. Query with product scope

In [ ]:
# Scoping by product improves relevance scoring
req_scoped = AgentRequest(
    query='What are the quality standards?',
    data_products=['cac'],
)
result = agent.execute(req_scoped)

print('Documents found:')
for k in result.data.get('knowledge', []):
    print(f"  - [{k['topic']}] (source: {k['source']})")

## 5. Knowledge documents available in mock store

In [ ]:
from services.pgvector.mock import _DOCS

print('Documents available in mock knowledge base:')
print('-' * 60)
for title, product, content in _DOCS:
    print(f'Title   : {title}')
    print(f'Product : {product}')
    print(f'Content : {content[:100]}...')
    print()